In [1]:
from peft import LoraConfig

# LoraConfig()

c:\PycharmProjects\projects\fine_tune_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0424 16:44:11.046000 58136 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B/")

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 879.10it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
for name, module in model.named_modules():
    print('当前模块的名字是：',name)
    print('当前模块是,',module)

当前模块的名字是： 
当前模块是, Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_fn): Func()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((

In [3]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

In [4]:
from peft import get_peft_model

lora_model = get_peft_model(model=model,peft_config=lora_config)


In [ ]:
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "logs/Qwen3-0.6B-TRL-LoRA"
# 1、处理数据，处理成SFTTrainer能够接收的数据类型
from datasets import load_dataset
dataset = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})

from typing import List,Dict
def convert_to_messages_format(examples:Dict[str,List]):
    conversations:List[List[Dict]] = examples["conversation"]
    result = []
    for conversation in conversations:
        # 遍历一个batch当中的一条数据
        all_messages:Dict = conversation[0]
        new_message_list = []
        new_message_list.append({"role":"user","content":all_messages["human"]})
        new_message_list.append({"role":"assistant","content":all_messages["assistant"]})
        result.append(new_message_list)
    
    return {"messages":result}


mapped_dataset = dataset.map(convert_to_messages_format,batched=True,remove_columns=['conversation_id', 'category', 'conversation', 'dataset'])

# 2、构造SFTConfig实例
config = SFTConfig(
    output_dir="./finetuned/Qwen3-0.6B-TRL-LoRA",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=12,
    #num_train_epochs=1,
    max_steps=1000,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    # warmup_steps = ,
    bf16=True,
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=100,
    save_steps=100,
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=100,
    max_length=2500,
    assistant_only_loss=True,
    report_to = ["tensorboard"],
    chat_template_path="./chat_template.jinja",
    load_best_model_at_end=True,
    greater_is_better=False,
    metric_for_best_model="eval_loss",
    save_total_limit=3
)

from transformers import AutoModelForCausalLM, AutoTokenizer
# 3、加载模型和tokenizer
tokenizer = AutoTokenizer.from_pretrained(r"model/Qwen3-0.6B/")

# 4、构造SFTTrainer实例
trainer = SFTTrainer(
    model=lora_model,
    processing_class=tokenizer,
    args=config,
    train_dataset=mapped_dataset["train"],
    eval_dataset=mapped_dataset["test"]
)


# 6、调用SFTTrainer的train进行训练
trainer.train()

# 7、调用SFTTrainer实例去保存模型参数和Tokenizer
trainer.save_model("./finetuned/Qwen3-0.6B-TRL-LoRA")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Truncating eval dataset: 100%|██████████| 500/500 [00:00<00:00, 27452.87 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
